# Ndizi Gemma-4 → .litertlm (ASR + Chat)

Converts the fine-tuned `smutuvi/gemma-4-e2b-sw-asr-ndizi` adapter to a `.litertlm` file  
ready for on-device deployment on **iPhone** and **Android**, preserving both:  
- 🎙 **Swahili ASR** (audio transcription)  
- 💬 **Chat conversation** (text generation)  

**Runtime required:** GPU (T4 or better)  
**Estimated time:** ~25–40 min on a free Colab T4  

### Before you start
1. Accept the Gemma license at https://huggingface.co/google/gemma-4-E2B-it
2. Create a HF token at https://huggingface.co/settings/tokens (read + write)
3. Set `HF_TOKEN` in the cell below

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
HF_TOKEN      = "hf_YOUR_TOKEN_HERE"   # ← paste your token
HF_USERNAME   = "smutuvi"              # ← your HF username
ADAPTER_REPO  = "smutuvi/gemma-4-e2b-sw-asr-ndizi"
BASE_MODEL_ID = "google/gemma-4-E2B-it"
MERGED_DIR    = "/content/ndizi_merged"
OUTPUT_DIR    = "/content/ndizi_litertlm"
MERGED_REPO   = f"{HF_USERNAME}/ndizi-gemma4-merged"  # private HF repo for conversion

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("Config set.")

In [ ]:
# ── Check GPU + disk ───────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

In [ ]:
# ── Install Python dependencies ────────────────────────────────────────────
# Note: we use --quiet to suppress noise, remove it if you want verbose logs
!pip install -q transformers==4.52.0 peft accelerate huggingface_hub soundfile
!pip install -q bitsandbytes  # optional: 8-bit quantisation for low-VRAM loading
print("Python packages installed.")

In [ ]:
# ── Install uv + litert-torch (conversion tool) ────────────────────────────
!curl -LsSf https://astral.sh/uv/install.sh | sh
import subprocess, os
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]
!uv tool install litert-torch-nightly
!uv tool install litert-lm
print("litert-torch and litert-lm installed.")

In [ ]:
# ── HuggingFace login ──────────────────────────────────────────────────────
from huggingface_hub import login, HfApi, list_repo_files
login(token=HF_TOKEN)
print("Logged in to HuggingFace.")

In [ ]:
# ── Detect: adapter or full model? ────────────────────────────────────────
try:
    files = list(list_repo_files(ADAPTER_REPO, token=HF_TOKEN))
    is_adapter = "adapter_config.json" in files
    print(f"Files in {ADAPTER_REPO}:")
    for f in files:
        print(f"  {f}")
    print(f"\nDetected as: {'PEFT/LoRA ADAPTER' if is_adapter else 'FULL MODEL'}")
except Exception as e:
    print(f"Could not list files (private repo?): {e}")
    is_adapter = True  # assume adapter — the merge step is safe either way
    print("Assuming adapter — will attempt merge.")

In [ ]:
# ── Merge LoRA adapter into base model ────────────────────────────────────
# Skip this cell if is_adapter was False above.
import torch, shutil
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

merged_path = Path(MERGED_DIR)

if merged_path.exists() and any(merged_path.iterdir()):
    print(f"{MERGED_DIR} already exists — skipping merge. Delete the folder to re-run.")
else:
    print(f"Loading base model: {BASE_MODEL_ID} ...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN,
    )

    print(f"Loading adapter: {ADAPTER_REPO} ...")
    model = PeftModel.from_pretrained(model, ADAPTER_REPO, token=HF_TOKEN)

    print("Merging weights ...")
    model = model.merge_and_unload()

    merged_path.mkdir(parents=True, exist_ok=True)
    print(f"Saving merged model to {MERGED_DIR} ...")
    model.save_pretrained(MERGED_DIR, safe_serialization=True)

    print("Saving tokenizer ...")
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN).save_pretrained(MERGED_DIR)

    # Copy auxiliary config files from base model (needed for audio/vision)
    from huggingface_hub import hf_hub_download
    for fname in ["preprocessor_config.json", "special_tokens_map.json",
                  "generation_config.json", "processor_config.json"]:
        try:
            src = hf_hub_download(BASE_MODEL_ID, fname, token=HF_TOKEN)
            shutil.copy(src, merged_path / fname)
            print(f"  Copied {fname}")
        except Exception:
            pass

    print("\n✓ Merge complete!")
    !ls -lh {MERGED_DIR}

In [ ]:
# ── Sanity check 1: Chat capability ───────────────────────────────────────
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    max_new_tokens=80,
)
prompt = "Wewe ni msaidizi wa lugha ya Kiswahili. Niambie habari za hali ya hewa leo."
output = pipe(prompt)[0]["generated_text"]
print("=== Chat test ===")
print(f"Prompt : {prompt}")
print(f"Output : {output}")
print("\n✓ Chat capability confirmed.")

In [ ]:
# ── Sanity check 2: ASR pipeline loads ────────────────────────────────────
# This verifies the audio sub-model loads correctly.
# If you have a real WAV file, set AUDIO_PATH below.
import wave, tempfile, torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

AUDIO_PATH = None  # e.g. "/content/sample_swahili.wav" — leave None for silence test

if AUDIO_PATH is None:
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    with wave.open(tmp.name, "w") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(16000)
        wf.writeframes(b"\x00" * 16000 * 2)  # 1 second silence
    AUDIO_PATH = tmp.name
    print(f"Using synthetic 1-second silence file: {AUDIO_PATH}")

try:
    processor = AutoProcessor.from_pretrained(MERGED_DIR)
    asr_model = Gemma3ForConditionalGeneration.from_pretrained(
        MERGED_DIR, torch_dtype=torch.float16, device_map="auto"
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "audio", "audio": AUDIO_PATH},
            {"type": "text",  "text": "Andika maneno unayosikia katika sauti hii."},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(asr_model.device)
    with torch.no_grad():
        out_ids = asr_model.generate(**inputs, max_new_tokens=80)
    print(f"ASR output: {processor.decode(out_ids[0], skip_special_tokens=True)!r}")
    print("✓ ASR pipeline loads and runs correctly.")
    del asr_model  # free VRAM before conversion
except Exception as e:
    print(f"⚠ ASR test error: {e}")
    print("  (This may still be OK — audio sub-model loads separately in LiteRT-LM)")

In [ ]:
# ── Push merged model to HuggingFace (private) ────────────────────────────
# litert-torch works most reliably from an HF repo ID.
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(MERGED_REPO, exist_ok=True, private=True)
print(f"Uploading merged model to {MERGED_REPO} ...")
api.upload_folder(folder_path=MERGED_DIR, repo_id=MERGED_REPO, token=HF_TOKEN)
print(f"✓ Pushed to https://huggingface.co/{MERGED_REPO}")

In [ ]:
# ── Convert to .litertlm ──────────────────────────────────────────────────
# --externalize_embedder  → memory-maps 1.12 GB embedding table on device
#                           (essential for iPhone — drops working RAM to ~607 MB)
# The audio + vision sub-models are included automatically for multimodal Gemma-4.

import subprocess, os
os.makedirs(OUTPUT_DIR, exist_ok=True)
env = os.environ.copy()
env["PATH"] = os.environ["HOME"] + "/.local/bin:" + env["PATH"]

cmd = [
    "litert-torch", "export_hf",
    f"--model={MERGED_REPO}",
    f"--output_dir={OUTPUT_DIR}",
    "--externalize_embedder",
]
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, env=env, capture_output=False, text=True)
if result.returncode != 0:
    print("\n⚠ Conversion failed — trying local path fallback...")
    cmd[2] = f"--model={MERGED_DIR}"
    subprocess.run(cmd, env=env, check=True)

!ls -lh {OUTPUT_DIR}
print("\n✓ Conversion complete!")

In [ ]:
# ── Local inference test with litert-lm ───────────────────────────────────
import subprocess, os, glob
env = os.environ.copy()
env["PATH"] = os.environ["HOME"] + "/.local/bin:" + env["PATH"]

models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
if not models:
    print("No .litertlm found — check conversion step above.")
else:
    model_file = models[0]
    print(f"Testing: {model_file}")
    subprocess.run(
        ["litert-lm", "run", model_file, "--prompt=Habari, unaweza kunisaidia kwa Kiswahili?"],
        env=env, check=True
    )
    print("\n✓ Chat inference works with litert-lm.")

In [ ]:
# ── Install LiteRT-LM Python API (needed for ASR testing) ─────────────────
# The CLI only supports text prompts; the Python API supports audio input too.
!pip install -q litert-lm-api-nightly
print("litert-lm Python API installed.")

In [ ]:
# ── Test 1: Chat via LiteRT-LM Python API ────────────────────────────────
# This tests the .litertlm file directly — same runtime as the iPhone.
import glob, litert_lm

models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
assert models, f"No .litertlm in {OUTPUT_DIR} — run conversion cell first."
MODEL_PATH = models[0]
print(f"Using model: {MODEL_PATH}")

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation(
        messages=[{"role": "system", "content": [{"type": "text",
            "text": "Wewe ni msaidizi wa lugha ya Kiswahili. Jibu kwa Kiswahili."}]}
        ]
    ) as conv:
        prompts = [
            "Habari yako?",
            "Neno 'kompyuta' linamaanisha nini?",
        ]
        for p in prompts:
            print(f"\nUser : {p}")
            resp = conv.send_message(p)
            print(f"Model: {resp['content'][0]['text']}")

print("\n✓ Chat test complete.")

In [ ]:
# ── Upload a Swahili audio file for ASR testing ───────────────────────────
# Upload a .wav file from your computer, OR skip this cell to use a
# synthetic silence file (just verifies the pipeline works).
#
# Audio requirements for Gemma-4 audio encoder:
#   - Format  : WAV (PCM, mono or stereo)
#   - Sample rate: 16 000 Hz recommended
#   - Duration: up to ~30 seconds
#
# To record a quick test clip on your phone:
#   iOS Voice Memos → Export as M4A → convert to WAV:
#   ffmpeg -i recording.m4a -ar 16000 -ac 1 recording.wav

from google.colab import files as colab_files
import os

print("Select a WAV file to upload (or press Cancel to use silence placeholder)")
try:
    uploaded = colab_files.upload()
    AUDIO_TEST_PATH = list(uploaded.keys())[0]
    print(f"Uploaded: {AUDIO_TEST_PATH}  ({os.path.getsize(AUDIO_TEST_PATH)/1024:.1f} KB)")
except Exception:
    # Fallback: generate 3 seconds of silence
    import wave, tempfile
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    with wave.open(tmp.name, "w") as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(16000)
        wf.writeframes(b"\x00" * 16000 * 2 * 3)  # 3 seconds silence
    AUDIO_TEST_PATH = tmp.name
    print(f"Using silence placeholder: {AUDIO_TEST_PATH}")

In [ ]:
# ── Test 2: ASR (audio transcription) via LiteRT-LM Python API ───────────
# audio_backend=Backend.CPU enables the audio sub-model.
# GPU audio backend is not yet available in LiteRT-LM Python API.
import glob, litert_lm

models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
MODEL_PATH = models[0]

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

print(f"Loading model with audio backend: {MODEL_PATH}")
with litert_lm.Engine(
    MODEL_PATH,
    audio_backend=litert_lm.Backend.CPU,   # ← enables ASR sub-model
) as engine:
    with engine.create_conversation() as conv:
        # ASR prompt: ask the model to transcribe
        asr_message = {
            "role": "user",
            "content": [
                {"type": "audio", "path": AUDIO_TEST_PATH},
                {"type": "text",  "text": "Andika maneno unayosikia katika sauti hii."},
            ],
        }
        print("\nRunning ASR inference...")
        response = conv.send_message(asr_message)
        transcript = response["content"][0]["text"]
        print(f"\nTranscription:\n{transcript}")

print("\n✓ ASR test complete.")

In [ ]:
# ── Test 3: Combined — transcribe then ask a follow-up question ───────────
# This simulates a real Ndizi use-case: transcribe audio, then chat about it.
import glob, litert_lm

models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
MODEL_PATH = models[0]

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

with litert_lm.Engine(
    MODEL_PATH,
    audio_backend=litert_lm.Backend.CPU,
) as engine:
    with engine.create_conversation(
        messages=[{"role": "system", "content": [{"type": "text",
            "text": "Wewe ni msaidizi wa lugha ya Kiswahili anayeweza kusikiliza na kuongea."}]}
        ]
    ) as conv:

        # Turn 1: transcribe audio
        print("--- Turn 1: ASR ---")
        r1 = conv.send_message({
            "role": "user",
            "content": [
                {"type": "audio", "path": AUDIO_TEST_PATH},
                {"type": "text",  "text": "Andika kwa usahihi maneno uliyosikia."},
            ],
        })
        print(f"Transcript: {r1['content'][0]['text']}")

        # Turn 2: follow-up chat using the same conversation context
        print("\n--- Turn 2: Chat follow-up ---")
        r2 = conv.send_message("Fanya muhtasari mfupi wa maneno hayo.")
        print(f"Summary  : {r2['content'][0]['text']}")

print("\n✓ Combined ASR + Chat test complete.")

In [ ]:
# ── Download the .litertlm file ────────────────────────────────────────────
# Option A: download directly to your computer via Colab
import glob
from google.colab import files
models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
if models:
    print(f"File size: {os.path.getsize(models[0]) / (1024**3):.2f} GB")
    print("Initiating download... (this may take a few minutes)")
    files.download(models[0])
else:
    print("No .litertlm found.")

In [ ]:
# ── Option B: push .litertlm to your HF repo for download ─────────────────
# This is better for large files (>2 GB) — download from HF instead of Colab
import glob
from huggingface_hub import HfApi

LITERTLM_REPO = f"{HF_USERNAME}/ndizi-gemma4-litertlm"  # new repo for the final file
api = HfApi(token=HF_TOKEN)
api.create_repo(LITERTLM_REPO, exist_ok=True, private=True)

models = glob.glob(f"{OUTPUT_DIR}/*.litertlm")
if models:
    print(f"Uploading {models[0]} to {LITERTLM_REPO} ...")
    api.upload_file(
        path_or_fileobj=models[0],
        path_in_repo="model.litertlm",
        repo_id=LITERTLM_REPO,
        token=HF_TOKEN,
    )
    print(f"✓ Download from: https://huggingface.co/{LITERTLM_REPO}")

## Next: Deploy on iPhone

### Quick test (no code)
1. Download the `.litertlm` file to your iPhone (Files app or AirDrop)
2. Install **Google AI Edge Gallery** from the App Store
3. Open it → Load model → select your `.litertlm` file
4. Test both chat and audio input

### Build into your iOS app
Add this Swift Package in Xcode: `https://github.com/google-ai-edge/LiteRT-LM`

```swift
import LiteRTLM

// Initialize with BOTH text and audio backends
let config = try EngineConfig(
    modelPath: Bundle.main.path(forResource: "model", ofType: "litertlm")!,
    backend: .gpu,            // Metal GPU for text decoding
    audioBackend: .cpu(),     // CPU audio executor for ASR
    cacheDir: NSTemporaryDirectory()
)
let engine = Engine(engineConfig: config)
try await engine.initialize()
let conversation = try await engine.createConversation()

// Chat
let reply = try await conversation.sendMessage(Message("Habari?"))
print(reply.toString)

// ASR transcription
let audioMsg = Message(contents: [
    Content.audioFile("/path/to/recording.wav"),
    Content.text("Andika maneno unayosikia.")
])
let transcript = try await conversation.sendMessage(audioMsg)
print(transcript.toString)
```

> **Tip:** If the GPU backend crashes, change `backend: .gpu` to `backend: .cpu()` as a diagnostic.  
> The Swift API is Early Preview (v0.12.0) — GPU issues are common on older iPhones.